### 0. Imports and Paths

In [1]:
# import packages
from pathlib import Path
import random
import shutil

### 1. Define dataset paths

In [2]:
# original dataset paths
SRC_ROOT = Path("../data")

SRC_IMAGES_DIR = SRC_ROOT / "images"
SRC_LABELS_DIR = SRC_ROOT / "labels"

TRAIN_FILE = SRC_ROOT / "train_files.txt"
VAL_FILE = SRC_ROOT / "val_files.txt"

In [3]:
# yolo dataset paths
DST_ROOT = Path("../data_for_yolo")

DIRS = {
    "train": {
        "images": DST_ROOT / "images" / "train",
        "labels": DST_ROOT / "labels" / "train",
    },
    "val": {
        "images": DST_ROOT / "images" / "val",
        "labels": DST_ROOT / "labels" / "val",
    },
    "test": {
        "images": DST_ROOT / "images" / "test",
        "labels": DST_ROOT / "labels" / "test",
    }
}

### 2. Create directories

In [4]:
for split in DIRS.values():
    split["images"].mkdir(parents=True, exist_ok=True)
    split["labels"].mkdir(parents=True, exist_ok=True)

### 3. Splitting images and labels for YOLO

In [5]:
# Helper function
def read_file_list(file_path):
    with open(file_path) as f:
        return [line.strip() for line in f.readlines()]

In [6]:
# Load splits
train_files = read_file_list(TRAIN_FILE)
test_files = read_file_list(VAL_FILE)  # original 20% becomes TEST

In [7]:
# Split train -> train + val
random.seed(42)  # reproducibility

random.shuffle(train_files)

split_index = int(len(train_files) * 0.875)  # 70/10/20 total

new_train = train_files[:split_index]
new_val = train_files[split_index:]

splits = {
    "train": new_train,
    "val": new_val,
    "test": test_files,
}

In [8]:
# Copy files
def copy_pair(filename, split):

    filename = Path(filename).name  # remove "images/"
    stem = Path(filename).stem

    img_src = SRC_IMAGES_DIR / filename
    lbl_src = SRC_LABELS_DIR / (stem + ".txt")

    img_dst = DIRS[split]["images"] / filename
    lbl_dst = DIRS[split]["labels"] / (stem + ".txt")

    if not img_src.exists():
        print(f"Missing image: {img_src}")
        return

    shutil.copy(img_src, img_dst)

    if lbl_src.exists():
        shutil.copy(lbl_src, lbl_dst)
    else:
        print(f"Missing label: {lbl_src}")


for split, files in splits.items():
    print(f"Processing {split}: {len(files)} files")

    for fname in files:
        copy_pair(fname, split)

print("Dataset prepared successfully.")

Processing train: 5669 files
Processing val: 810 files
Processing test: 1620 files
Dataset prepared successfully.


### 3. Check if splitting was succesful

In [9]:
ROOT = Path("../data_for_yolo")

splits = ["train", "val", "test"]

all_images = set()

for split in splits:
    
    img_dir = ROOT / "images" / split
    lbl_dir = ROOT / "labels" / split
    
    images = {p.stem for p in img_dir.glob("*")}
    labels = {p.stem for p in lbl_dir.glob("*.txt")}
    
    all_images |= images
    
    print(f"\n--- {split.upper()} ---")
    print(f"Images: {len(images)}")
    print(f"Labels: {len(labels)}")
    
    print(f"Images without labels: {len(images - labels)}")
    print(f"Labels without images: {len(labels - images)}")

# dataset totals
print("\n--- DATASET TOTAL ---")
print(f"Total images: {len(all_images)}")

train_images = {p.stem for p in (ROOT/"images/train").glob("*")}
val_images   = {p.stem for p in (ROOT/"images/val").glob("*")}
test_images  = {p.stem for p in (ROOT/"images/test").glob("*")}

print(f"Train images: {len(train_images)} ({len(train_images)/len(all_images)*100:.1f}%)")
print(f"Val images: {len(val_images)} ({len(val_images)/len(all_images)*100:.1f}%)")
print(f"Test images: {len(test_images)} ({len(test_images)/len(all_images)*100:.1f}%)")

print(f"\nTrain/Val overlap: {len(train_images & val_images)}")
print(f"Train/Test overlap: {len(train_images & test_images)}")
print(f"Val/Test overlap: {len(val_images & test_images)}")


--- TRAIN ---
Images: 5669
Labels: 5669
Images without labels: 0
Labels without images: 0

--- VAL ---
Images: 810
Labels: 810
Images without labels: 0
Labels without images: 0

--- TEST ---
Images: 1620
Labels: 1620
Images without labels: 0
Labels without images: 0

--- DATASET TOTAL ---
Total images: 8099
Train images: 5669 (70.0%)
Val images: 810 (10.0%)
Test images: 1620 (20.0%)

Train/Val overlap: 0
Train/Test overlap: 0
Val/Test overlap: 0
